# RetailMax: Evidence-Based Prompt Improvement Exercise

## Learning objective
Start with a broad **Business-to-Data** prompt, inspect the RetailMax customer dataset, and improve the prompt in six phases:

1. Time horizon  
2. Retail channel  
3. Available data  
4. Decision constraints  
5. Expected output format  
6. Avoid assumptions

The notebook deliberately separates three sources of prompt context:

- **Evidence from the dataset**
- **Information not available in the dataset**
- **Managerial choices that must be supplied by the decision-maker**


## Activity workflow

1. Read the baseline prompt before inspecting the data.
2. Run each analysis phase.
3. Record what the dataset supports and what it does not support.
4. Observe how the prompt becomes more precise after each phase.
5. Copy the final prompt into an AI assistant.
6. Evaluate whether the output is more specific, testable, and decision-relevant than the baseline output.

In [1]:
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

## Baseline prompt

Run the next cell to display the same baseline prompt for every participant. Do **not** improve it yet.

In [30]:
baseline_prompt = """Act as a retail analytics consultant. RetailMax is experiencing declining profitability despite steady sales growth.

Translate this concern into:
- The managerial decision required
- Relevant KPIs
- Unit of analysis
- Required data
- Potential drivers
- Suitable analytical approaches
- Expected business actions

Present the answer as a Business-to-Data Canvas."""

display(Markdown("### Baseline prompt"))
print(baseline_prompt)

### Baseline prompt

Act as a retail analytics consultant. RetailMax is experiencing declining profitability despite steady sales growth.

Translate this concern into:
- The managerial decision required
- Relevant KPIs
- Unit of analysis
- Required data
- Potential drivers
- Suitable analytical approaches
- Expected business actions

Present the answer as a Business-to-Data Canvas.


### Baseline reflection
Before opening the data, note two weaknesses in the baseline prompt:

- What important business context is missing?
- Which parts of a likely AI response may become generic or speculative?

In [31]:
# Optional participant notes
baseline_weakness_1 = ""
baseline_weakness_2 = ""
print("1.", baseline_weakness_1 or "Add your observation")
print("2.", baseline_weakness_2 or "Add your observation")

1. Add your observation
2. Add your observation


## Load the RetailMax data

Place `RetailMax_Customer_Data.csv` in the same folder as this notebook. The loader also checks `/mnt/data` and accepts an Excel version if needed.

In [32]:
possible_files = [
    Path("RetailMax_Customer_Data.csv"),
    Path("/mnt/data/RetailMax_Customer_Data.csv")
]

data_path = next((p for p in possible_files if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "RetailMax data not found. Put RetailMax_Customer_Data.csv in the notebook folder."
    )

if data_path.suffix.lower() == ".csv":
    df = pd.read_csv(data_path)
else:
    df = pd.read_excel(data_path, engine="openpyxl")

# Normalize escaped or accidental spaces in column names.
df.columns = (
    df.columns.astype(str)
      .str.replace("\\_", "_", regex=True)
      .str.strip()
      .str.replace(" ", "_", regex=False)
)

print(f"Loaded: {data_path.name}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())

Loaded: RetailMax_Customer_Data.csv
Rows: 800 | Columns: 15


,Customer_ID,Age,Annual_Income_INR,Region,Preferred_Channel,Loyalty_Status,Primary_Category,Purchase_Frequency,Average_Basket_Value_INR,Annual_Spend_INR,Average_Discount_Rate,Return_Rate,Customer_Satisfaction_10,Estimated_Gross_Profit_INR,Estimated_Profit_Margin
0,RM0001,37.0,571000.0,West,Online,Platinum,Grocery,14,3011.52,39314.28,21.9,10.9,9.3,6491.12,16.5
1,RM0002,40.0,996000.0,North,Omnichannel,Silver,Grocery,7,1475.50,10898.01,16.7,19.7,5.4,1549.84,14.2
2,RM0003,38.0,523000.0,North,Store,NaN,Grocery,20,1497.42,29015.26,6.9,6.8,6.2,6469.93,22.3
3,RM0004,18.0,1051000.0,North,Store,Silver,Grocery,48,2040.00,95467.31,8.4,13.8,8.7,18101.56,19.0
4,RM0005,53.0,279000.0,West,Store,NaN,Fashion,3,1353.56,4237.77,13.4,5.0,7.1,1562.33,36.9


# Phase 1: Time horizon

**Question:** Over what period is profitability declining?

A time horizon can only be established if the data includes a valid date or period field. The code searches for likely time columns rather than inventing a period.

In [33]:
time_keywords = ("date", "year", "month", "quarter", "week", "period", "time")
time_columns = [c for c in df.columns if any(k in c.lower() for k in time_keywords)]

if time_columns:
    print("Potential time columns:", time_columns)
    for col in time_columns:
        parsed = pd.to_datetime(df[col], errors="coerce")
        if parsed.notna().any():
            print(f"{col}: {parsed.min().date()} to {parsed.max().date()}")
else:
    print("No date or period column was found.")
    print("Conclusion: the dataset cannot verify when profitability declined or whether sales grew over time.")

No date or period column was found.
Conclusion: the dataset cannot verify when profitability declined or whether sales grew over time.


In [34]:
if time_columns:
    phase1_context = "Use the period covered by the valid date field(s) identified in the dataset."
else:
    phase1_context = (
        "The supplied customer-level snapshot contains no date or period variable. "
        "Do not infer a time horizon or claim that a profitability trend has been verified from this file."
    )

prompt_phase_1 = baseline_prompt + "\n Time-horizon evidence: - " + phase1_context

display(Markdown("### Prompt after Phase 1"))
print(prompt_phase_1)

### Prompt after Phase 1

Act as a retail analytics consultant. RetailMax is experiencing declining profitability despite steady sales growth.

Translate this concern into:
- The managerial decision required
- Relevant KPIs
- Unit of analysis
- Required data
- Potential drivers
- Suitable analytical approaches
- Expected business actions

Present the answer as a Business-to-Data Canvas.
 Time-horizon evidence: - The supplied customer-level snapshot contains no date or period variable. Do not infer a time horizon or claim that a profitability trend has been verified from this file.


# Phase 2: Retail channel

**Question:** Which retail channels are represented, and how many customer records appear in each channel?

This phase converts observed channel values into prompt context.

In [35]:
channel_candidates = [c for c in df.columns if "channel" in c.lower()]
print("Channel fields found:", channel_candidates)

if channel_candidates:
    channel_col = channel_candidates[0]
    channel_summary = (
        df[channel_col]
        .fillna("Missing")
        .value_counts(dropna=False)
        .rename_axis(channel_col)
        .reset_index(name="Customer_Records")
    )
    channel_summary["Share_%"] = (channel_summary["Customer_Records"] / len(df) * 100).round(1)
    display(channel_summary)
    observed_channels = [str(x) for x in df[channel_col].dropna().unique()]
else:
    channel_col = None
    observed_channels = []
    print("No channel field was found.")

Channel fields found: ['Preferred_Channel']


,Preferred_Channel,Customer_Records,Share_%
0,Store,358,44.8
1,Online,258,32.2
2,Omnichannel,184,23.0


In [36]:
if observed_channels:
    phase2_context = (
        f"The dataset contains the channel field '{channel_col}' with observed values: "
        + ", ".join(sorted(observed_channels)) + "."
    )
else:
    phase2_context = "No retail-channel variable was found; do not make channel comparisons."

prompt_phase_2 = prompt_phase_1 + "\n Retail-channel evidence: - " + phase2_context

display(Markdown("### Prompt after Phase 2"))
print(prompt_phase_2)

### Prompt after Phase 2

Act as a retail analytics consultant. RetailMax is experiencing declining profitability despite steady sales growth.

Translate this concern into:
- The managerial decision required
- Relevant KPIs
- Unit of analysis
- Required data
- Potential drivers
- Suitable analytical approaches
- Expected business actions

Present the answer as a Business-to-Data Canvas.
 Time-horizon evidence: - The supplied customer-level snapshot contains no date or period variable. Do not infer a time horizon or claim that a profitability trend has been verified from this file.
 Retail-channel evidence: - The dataset contains the channel field 'Preferred_Channel' with observed values: Omnichannel, Online, Store.


# Phase 3: Available data

**Question:** What can actually be analyzed?

Participants examine fields, data types, missingness, and uniqueness. This prevents the prompt from requesting analyses that the file cannot support.

In [37]:
profile = pd.DataFrame({
    "Variable": df.columns,
    "Data_Type": [str(df[c].dtype) for c in df.columns],
    "Non_Missing": [int(df[c].notna().sum()) for c in df.columns],
    "Missing": [int(df[c].isna().sum()) for c in df.columns],
    "Missing_%": [round(df[c].isna().mean() * 100, 1) for c in df.columns],
    "Unique_Values": [int(df[c].nunique(dropna=True)) for c in df.columns],
})
display(profile)

,Variable,Data_Type,Non_Missing,Missing,Missing_%,Unique_Values
0,Customer_ID,object,800,0,0.0,800
1,Age,float64,741,59,7.4,56
2,Annual_Income_INR,float64,731,69,8.6,555
3,Region,object,800,0,0.0,5
4,Preferred_Channel,object,800,0,0.0,3
5,Loyalty_Status,object,589,211,26.4,3
6,Primary_Category,object,800,0,0.0,5
7,Purchase_Frequency,int64,800,0,0.0,56
8,Average_Basket_Value_INR,float64,800,0,0.0,799
9,Annual_Spend_INR,float64,800,0,0.0,800


In [38]:
roles = {
    "Identifier": [c for c in df.columns if c.lower().endswith("id") or "_id" in c.lower()],
    "Segmentation dimensions": [c for c in df.columns if any(k in c.lower() for k in ["region", "channel", "loyalty", "category"])],
    "Customer characteristics": [c for c in df.columns if any(k in c.lower() for k in ["age", "income"])],
    "Behavior and value": [c for c in df.columns if any(k in c.lower() for k in ["frequency", "basket", "spend"])],
    "Potential profitability drivers": [c for c in df.columns if any(k in c.lower() for k in ["discount", "return", "satisfaction"])],
    "Profitability outcomes": [c for c in df.columns if any(k in c.lower() for k in ["profit", "margin"])],
}

role_table = pd.DataFrame(
    [(role, ", ".join(cols) if cols else "Not available") for role, cols in roles.items()],
    columns=["Analytical_Role", "Variables"]
)
display(role_table)

,Analytical_Role,Variables
0,Identifier,Customer_ID
1,Segmentation dimensions,"Region, Preferred_Channel, Loyalty_Status, Primary_Category"
2,Customer characteristics,"Age, Annual_Income_INR, Average_Basket_Value_INR, Average_Discount_Rate"
3,Behavior and value,"Purchase_Frequency, Average_Basket_Value_INR, Annual_Spend_INR"
4,Potential profitability drivers,"Average_Discount_Rate, Return_Rate, Customer_Satisfaction_10"
5,Profitability outcomes,"Estimated_Gross_Profit_INR, Estimated_Profit_Margin"


In [39]:
available_list = ", ".join(df.columns)
phase3_context = (
    "Use only the following fields: " + available_list + ". "
    "Treat reported profit and margin fields as estimated measures where their names say 'Estimated'. "
    "Account for missing values before analysis."
)

prompt_phase_3 = prompt_phase_2 + "\n Available-data evidence:- " + phase3_context

display(Markdown("### Prompt after Phase 3"))
print(prompt_phase_3)

### Prompt after Phase 3

Act as a retail analytics consultant. RetailMax is experiencing declining profitability despite steady sales growth.

Translate this concern into:
- The managerial decision required
- Relevant KPIs
- Unit of analysis
- Required data
- Potential drivers
- Suitable analytical approaches
- Expected business actions

Present the answer as a Business-to-Data Canvas.
 Time-horizon evidence: - The supplied customer-level snapshot contains no date or period variable. Do not infer a time horizon or claim that a profitability trend has been verified from this file.
 Retail-channel evidence: - The dataset contains the channel field 'Preferred_Channel' with observed values: Omnichannel, Online, Store.
 Available-data evidence:- Use only the following fields: Customer_ID, Age, Annual_Income_INR, Region, Preferred_Channel, Loyalty_Status, Primary_Category, Purchase_Frequency, Average_Basket_Value_INR, Annual_Spend_INR, Average_Discount_Rate, Return_Rate, Customer_Satisfaction_10, Estimated_Gross_Pro

## Evidence check: what relationships are visible?

The following summaries do not prove causation. They help participants identify **testable hypotheses** that can be included in the improved prompt.

In [40]:
profit_col = "Estimated_Profit_Margin" if "Estimated_Profit_Margin" in df.columns else None
segment_cols = [c for c in ["Preferred_Channel", "Primary_Category", "Region", "Loyalty_Status"] if c in df.columns]

if profit_col:
    for col in segment_cols:
        summary = (
            df.groupby(col, dropna=False)[profit_col]
              .agg(Customer_Records="size", Mean_Profit_Margin="mean", Median_Profit_Margin="median")
              .sort_values("Mean_Profit_Margin")
              .round(2)
        )
        display(Markdown(f"### Profitability by {col}"))
        display(summary)
else:
    print("Estimated_Profit_Margin was not found.")

### Profitability by Preferred_Channel

,Customer_Records,Mean_Profit_Margin,Median_Profit_Margin
Preferred_Channel,,,
Omnichannel,184,24.49,23.45
Online,258,24.65,25.40
Store,358,25.65,26.05


### Profitability by Primary_Category

,Customer_Records,Mean_Profit_Margin,Median_Profit_Margin
Primary_Category,,,
Electronics,108,14.94,15.9
Grocery,219,18.39,18.6
Home,193,27.98,28.9
Beauty,111,30.03,30.9
Fashion,169,33.58,34.0


### Profitability by Region

,Customer_Records,Mean_Profit_Margin,Median_Profit_Margin
Region,,,
Central,125,24.16,23.5
North,164,24.93,25.6
South,187,25.17,26.2
West,167,25.38,25.1
East,157,25.45,26.4


### Profitability by Loyalty_Status

,Customer_Records,Mean_Profit_Margin,Median_Profit_Margin
Loyalty_Status,,,
Platinum,101,24.36,23.60
Silver,266,24.85,24.80
NaN,211,25.19,26.40
Gold,222,25.50,26.55


In [18]:
relationship_cols = [
    c for c in [
        "Average_Discount_Rate", "Return_Rate", "Customer_Satisfaction_10",
        "Purchase_Frequency", "Average_Basket_Value_INR", "Annual_Spend_INR",
        "Estimated_Profit_Margin"
    ] if c in df.columns
]

if len(relationship_cols) >= 2:
    correlations = df[relationship_cols].corr(numeric_only=True).round(2)
    display(correlations)
    print("Interpret correlations as associations, not causal effects.")

,Average_Discount_Rate,Return_Rate,Customer_Satisfaction_10,Purchase_Frequency,Average_Basket_Value_INR,Annual_Spend_INR,Estimated_Profit_Margin
Average_Discount_Rate,1.00,0.16,0.01,0.02,-0.01,-0.02,-0.45
Return_Rate,0.16,1.00,-0.13,0.05,-0.00,-0.05,-0.47
Customer_Satisfaction_10,0.01,-0.13,1.00,0.04,-0.01,0.03,0.07
Purchase_Frequency,0.02,0.05,0.04,1.00,-0.10,0.59,-0.04
Average_Basket_Value_INR,-0.01,-0.00,-0.01,-0.10,1.00,0.44,-0.02
Annual_Spend_INR,-0.02,-0.05,0.03,0.59,0.44,1.00,-0.00
Estimated_Profit_Margin,-0.45,-0.47,0.07,-0.04,-0.02,-0.00,1.00


Interpret correlations as associations, not causal effects.


# Phase 4: Decision constraints

**Important distinction:** Feasible actions cannot be discovered purely from a customer dataset. Budget, authority, operational capacity, pricing rules, and implementation timing are managerial inputs.

Participants should therefore specify constraints rather than allow AI to invent them.

In [41]:
# Edit these as per the assumptions made by you.
decision_constraints = {
    "Implementation horizon": "Recommendations should be implementable within six months.",
    "Pricing": "Do not recommend an immediate across-the-board price increase.",
    "Footprint": "Do not recommend closing stores based only on this customer-level snapshot.",
    "Evidence standard": "Recommend actions only when linked to measures available in the dataset.",
}

constraints_text = " ".join(f"{k}: {v}" for k, v in decision_constraints.items())
print(constraints_text)

Implementation horizon: Recommendations should be implementable within six months. Pricing: Do not recommend an immediate across-the-board price increase. Footprint: Do not recommend closing stores based only on this customer-level snapshot. Evidence standard: Recommend actions only when linked to measures available in the dataset.


In [43]:
prompt_phase_4 = prompt_phase_3 + "\n Decision constraints supplied by management:- " + constraints_text

display(Markdown("### Prompt after Phase 4"))
print(prompt_phase_4)

### Prompt after Phase 4

Act as a retail analytics consultant. RetailMax is experiencing declining profitability despite steady sales growth.

Translate this concern into:
- The managerial decision required
- Relevant KPIs
- Unit of analysis
- Required data
- Potential drivers
- Suitable analytical approaches
- Expected business actions

Present the answer as a Business-to-Data Canvas.
 Time-horizon evidence: - The supplied customer-level snapshot contains no date or period variable. Do not infer a time horizon or claim that a profitability trend has been verified from this file.
 Retail-channel evidence: - The dataset contains the channel field 'Preferred_Channel' with observed values: Omnichannel, Online, Store.
 Available-data evidence:- Use only the following fields: Customer_ID, Age, Annual_Income_INR, Region, Preferred_Channel, Loyalty_Status, Primary_Category, Purchase_Frequency, Average_Basket_Value_INR, Annual_Spend_INR, Average_Discount_Rate, Return_Rate, Customer_Satisfaction_10, Estimated_Gross_Pro

# Phase 5: Avoid assumptions

This phase explicitly lists claims the current dataset cannot support. It is the safeguard against confident but untestable recommendations.

In [45]:
likely_missing_domains = {
    "Time trend": not bool(time_columns),
    "Transaction-level chronology": not any("transaction" in c.lower() or "date" in c.lower() for c in df.columns),
    "Product-level margin": not any("product_id" in c.lower() or c.lower() == "product" for c in df.columns),
    "Inventory and stock-outs": not any("inventory" in c.lower() or "stock" in c.lower() for c in df.columns),
    "Marketing spend and campaign exposure": not any("marketing" in c.lower() or "campaign" in c.lower() for c in df.columns),
    "Competitor prices": not any("competitor" in c.lower() for c in df.columns),
    "Delivery or fulfilment cost": not any("delivery" in c.lower() or "fulfil" in c.lower() for c in df.columns),
}

missing_table = pd.DataFrame(
    [(domain, "Not supported by current fields" if missing else "Potentially supported")
     for domain, missing in likely_missing_domains.items()],
    columns=["Business_Claim", "Dataset_Status"]
)
display(missing_table)

,Business_Claim,Dataset_Status
0,Time trend,Not supported by current fields
1,Transaction-level chronology,Not supported by current fields
2,Product-level margin,Not supported by current fields
3,Inventory and stock-outs,Not supported by current fields
4,Marketing spend and campaign exposure,Not supported by current fields
5,Competitor prices,Not supported by current fields
6,Delivery or fulfilment cost,Not supported by current fields


In [46]:
unsupported = [domain for domain, missing in likely_missing_domains.items() if missing]
phase6_context = (
    "Do not assume or claim evidence about: " + ", ".join(unsupported) + ". "
    "Do not treat association as causation. Do not assume the stated decline or steady sales growth has been verified when no time field is present. "
    "Label untested explanations as hypotheses and identify the additional data needed to test them."
)

prompt_phase_6 = prompt_phase_4 + "\n Assumptions to avoid:- " + phase6_context

display(Markdown("### Prompt after Phase 6"))
print(prompt_phase_6)

### Prompt after Phase 6

Act as a retail analytics consultant. RetailMax is experiencing declining profitability despite steady sales growth.

Translate this concern into:
- The managerial decision required
- Relevant KPIs
- Unit of analysis
- Required data
- Potential drivers
- Suitable analytical approaches
- Expected business actions

Present the answer as a Business-to-Data Canvas.
 Time-horizon evidence: - The supplied customer-level snapshot contains no date or period variable. Do not infer a time horizon or claim that a profitability trend has been verified from this file.
 Retail-channel evidence: - The dataset contains the channel field 'Preferred_Channel' with observed values: Omnichannel, Online, Store.
 Available-data evidence:- Use only the following fields: Customer_ID, Age, Annual_Income_INR, Region, Preferred_Channel, Loyalty_Status, Primary_Category, Purchase_Frequency, Average_Basket_Value_INR, Annual_Spend_INR, Average_Discount_Rate, Return_Rate, Customer_Satisfaction_10, Estimated_Gross_Pro

# Final evidence-based prompt

The final version below combines all six improvements. Copy it into an AI assistant only after participants have reviewed the evidence generated above.

In [47]:
final_task = """

Task:
Translate the managerial concern into an analytical investigation. Identify:
1. The managerial decision required
2. Business questions that can be answered with the available fields
3. Relevant KPIs and precise definitions
4. Appropriate unit or units of analysis
5. Required variables from the supplied dataset
6. Testable hypotheses about profitability
7. Suitable descriptive and diagnostic analytical approaches
8. Additional data required for unsupported hypotheses
9. Feasible business actions, stated conditionally on possible findings

Do not provide a definitive root-cause conclusion before the analyses are performed.
"""

final_prompt = prompt_phase_6 + final_task

display(Markdown("## FINAL PROMPT"))
print(final_prompt)

## FINAL PROMPT

Act as a retail analytics consultant. RetailMax is experiencing declining profitability despite steady sales growth.

Translate this concern into:
- The managerial decision required
- Relevant KPIs
- Unit of analysis
- Required data
- Potential drivers
- Suitable analytical approaches
- Expected business actions

Present the answer as a Business-to-Data Canvas.
 Time-horizon evidence: - The supplied customer-level snapshot contains no date or period variable. Do not infer a time horizon or claim that a profitability trend has been verified from this file.
 Retail-channel evidence: - The dataset contains the channel field 'Preferred_Channel' with observed values: Omnichannel, Online, Store.
 Available-data evidence:- Use only the following fields: Customer_ID, Age, Annual_Income_INR, Region, Preferred_Channel, Loyalty_Status, Primary_Category, Purchase_Frequency, Average_Basket_Value_INR, Annual_Spend_INR, Average_Discount_Rate, Return_Rate, Customer_Satisfaction_10, Estimated_Gross_Pro

## Six-phase comparison table

Use this table during the debrief to show how the prompt evolved.

In [28]:
comparison = pd.DataFrame([
    ["Baseline", "Broad business concern only", baseline_prompt],
    ["1. Time horizon", "Checks whether time-based claims are supportable", prompt_phase_1],
    ["2. Retail channel", "Adds observed channel scope", prompt_phase_2],
    ["3. Available data", "Restricts analysis to actual variables", prompt_phase_3],
    ["4. Decision constraints", "Adds managerial feasibility limits", prompt_phase_4],
    ["5. Output format", "Makes the response executive-friendly and comparable", prompt_phase_5],
    ["6. Avoid assumptions", "Separates evidence, hypotheses, and information gaps", prompt_phase_6],
], columns=["Version", "Improvement", "Prompt"])

display(comparison[["Version", "Improvement"]])

,Version,Improvement
0,Baseline,Broad business concern only
1,1. Time horizon,Checks whether time-based claims are supportable
2,2. Retail channel,Adds observed channel scope
3,3. Available data,Restricts analysis to actual variables
4,4. Decision constraints,Adds managerial feasibility limits
5,5. Output format,Makes the response executive-friendly and comparable
6,6. Avoid assumptions,"Separates evidence, hypotheses, and information gaps"
